# Model avec Pipeline

In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import joblib

# from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# erreur indicator
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

from sklearn.preprocessing import MaxAbsScaler

# Tout nos model utiliser dans l'entrainemen
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import time

## Explication

### Objectif:
Nous allons donc ici faire un model d'IA qui prend en entrainer un item du magasin et qui est capable de nous dire, `probabilite de rachat`, `nombre de fois acheter par des personnes unique`, `les meilleures heures de ventes`, `les meilleurs jours de vente`, `le meilleur item avec`, `l'emplacement dans le panier`, `la quantite ecoule` ainsi qu'un score globale defini selon le calcul ci-dessous.

### Realisation:
Pour se faire nous avons realiser une Pipeline, cette pipeline nous permet de regrouper toutes nos donnees pour les tester sur differents type de models et de parametre.

La liste de nos models peut etre retrouve plus bas dans `Liste Models`.

##### /!\     A noter que pour des raisons de place et de clairete, nous ne montrerons uniquement les amelioration que pour notre information de probabilite d'achat, mais pour chacune de nos informations nous avons essayer plusieurs combinaisons pour trouver la meilleur et obtenir le resultat le plus fidele et coherent.
##### Neanmoins nos demarche de recherche sont disponible dans le fichier `reflexion.ipynb`, seul le code precis des precedent essai ni figure pas. /!\


##### Column:
Pour nos column voici la structure que nous utiliserons pour entrainer nos Model, en voici une partie non exaustive:
order_number ->
 * number_count -> nombre total de order_number
 * order_number -> l'ordre de la commande

order_dow ->
 * dow_mean -> moyenne du jour de la semaine
 * dow_count -. frequence d'achat
 * dow_std -> equart type du jour de la semaine

order_hour_of_day ->
 * order_hour_of_day -> moyenne heures d'achat

add_to_cart_order ->
 * add_to_cart_order -> moyenne de l'emplacement dans le panier

reordored ->
 * reordored -> Oui ou non

## Data intialisation

In [2]:
#ici c'est mon CSV de base
CSV_untouch = pd.read_csv('CSV_Cleaned.csv')

## Predictions:

### Pourcentage de rachat

#### Explication

Pour cette partie elle est issue de notre fichier `Model_IA_Type_Regression_Logistique.ipynb`, comme citee precedement nous utilsons ce fichier comme exemple entier, mais ici nous n'allojns reprendre que la partie selctionner des donnes et les envoyer dans notre Pipeline.

#### Code

In [16]:
#on va faire un DF avec uniquement les product_name et les user_id pour avoir une ligne par produit et autant le lignes que de produits acheter par un user_id
DF_entrainement = CSV_untouch.groupby(["product_name"]).agg(
    add_to_cart_order=("add_to_cart_order", 'mean'),
    order_total_vente=('order_id', "count"),
    order_number_max=('order_number', 'max'),
    order_number_min=('order_number', 'min'),
    order_number_mean=('order_number', 'mean'),
    dow_mean=('order_dow', 'mean')
).reset_index() 

#ici on va faire nos reponses donc on garde le meme ordre et on va juste prendre la moyenne de reordered
DF_reponse = CSV_untouch.groupby(["product_name"]).agg({
    "reordered": 'mean'
}).reset_index() 

In [17]:
# comme on est sur un model de LogisticRegression on doit donner soit des 0 soit 1 donc on coupe le reordored en 0 ou 1
# DF_reponse['target'] = (DF_reponse['reordered'] > 0.5).astype(int)

# ici on va merge nos deux DF, celui nettiyer qui contient nos column d'entrainement + celui qui contient nos reponse avec les 0 ou 1 
df_final = pd.merge(DF_entrainement, DF_reponse[['product_name', 'reordered']], on=['product_name'])
# on degage les vide 
df_final = df_final.dropna()


In [18]:
# on defini ici nos column issue du DF finale
X = df_final[['add_to_cart_order', 'order_total_vente', 'order_number_max', 'order_number_min', 'order_number_mean', 'dow_mean']]
y = df_final['reordered']

# on va separer nos data en 2, 70% pour 'apprendre et 30% pour le tester comme ca on a un meilleur resultat
X_train_prct_rachat, X_test_prct_rachat, y_train_prct_rachat, y_test_prct_rachat = train_test_split(X, y, test_size=0.3, random_state=42)


In [19]:
# ici on entraine nos donnes sur notre Pipeline
Test_Model(X_train_prct_rachat, X_test_prct_rachat, y_train_prct_rachat, y_test_prct_rachat, X, df_final, "Prct_rachat", "regression")


Prct_rachat Régression Logistique (Reg)
Modèle : Régression Logistique (Reg), Type Modèle : regression
 MAE (Erreur position) : 0.17
 Score R2              : 0.1952
Temps d'exécution : 0.01 secondes
----------------------------------------

Prct_rachat Random Forest Regressor
Modèle : Random Forest Regressor, Type Modèle : regression
 MAE (Erreur position) : 0.14
 Score R2              : 0.3638
Temps d'exécution : 0.31 secondes
----------------------------------------

Prct_rachat Gradient Boosting (Hist)
Modèle : Gradient Boosting (Hist), Type Modèle : regression
 MAE (Erreur position) : 0.14
 Score R2              : 0.3757
Temps d'exécution : 0.16 secondes
----------------------------------------
model save


{'Régression Logistique (Reg)': 0.19523814013333007,
 'Random Forest Regressor': 0.3637630361079496,
 'Gradient Boosting (Hist)': 0.37565133626853664}

### Nombre de fois acheter par des personnes unique

#### Explication:

Pour cette partie nous n'allons pas nous servir du model d'IA, exceptionnelement nous allons directement recuperer des donnees depuis notre CSV, car il est impossible avec les informations que nous possedons de predire efficacement et correctement le nombre d'acheteur unique.

(A noter que nous l'avons code ici mais en realite il est utilise et coder directement dans le streamlit)

#### Code

In [ ]:
# on va definir notre DF qui va contenir le nombre d'utilisateurs unique qui ont achetes se produits
CSV_Clean = CSV_untouch.groupby(["product_name"]).agg(
    user_id=('user_id', "nunique"),
).reset_index() 

### Meilleures heures

#### Explications:

Ici nous allons essayer de predire avec notre IA la meilleure heure de vente de se produits, cela nous permettra de connaitre l'heures dans la journee ou en moyenne tel ou tel items sont le plus vendu.

Pour les heures nous avons du realiser plusieurs test, nous avons tout d'abord tester des models de regression mais nous nous sommes vite delaisse pour atteidnre les models de classifications.
Apres plusieurs essaie et test notre precision n'etait que de 3 pourcent, il n'y avait aucune coleration directe entre nos heures seul et nos donness disponible, nous avons donc decider de separer nos heueres comme des saisons en 4 partie cite ci dessous pour faciliter l'apprentissage et augemnter la precision.

#### Code:

In [45]:
# On defini notre DF qui contient nos informations d'apprentisage
DF_Apprentissage = CSV_untouch.groupby(["product_name"]).agg(
    order_number=('order_number', 'max'),
    add_to_cart_order=('add_to_cart_order', 'mean'),
    dow_mean=('order_dow', 'mean'),
    order_total_vente=('order_id', "count"),
    user_id_unique=('user_id', 'nunique'),
    reordered_prct=('reordered', 'mean'),
    department=('department', 'first'),
    aisle=('aisle', 'first')
).reset_index() 


# on defini notre DF de reponse qui contient les heures
# ici on prend la Mode, la mode retourne les meilleurs heures de vente pour chaque produits, et nous allons seletionne la premiere
DF_Reponse = CSV_untouch.groupby("product_name")['order_hour_of_day'].agg(
    lambda x: x.mode()[0]
).reset_index()


# limnite pour indiquer qund on cut au niveau des heures
limites = [-1, 6, 12, 18, 24]
noms_periodes = ['Nuit', 'Matin', 'Apres-midi', 'Soir']

# on va ici cut nos data en fonction de leurs apartenances
DF_Reponse['periode'] = pd.cut(DF_Reponse['order_hour_of_day'], bins=limites, labels=noms_periodes)

In [46]:
# on defini nos X et Y qui vont etre utilise pour separer nos donnes
X = DF_Apprentissage[['order_number', 'add_to_cart_order', 'dow_mean', 'order_total_vente', 'user_id_unique', 'reordered_prct', 'department','aisle']]
y = DF_Reponse['periode']

# on defini ici toute nos valeurs d'entraienement et de test
X_train_heure, X_test_heure, y_train_heure, y_test_heure = train_test_split(X, y, test_size=0.3, random_state=42)

In [47]:
# on entraine notre model sur la pipeline
Test_Model(X_train_heure, X_test_heure, y_train_heure, y_test_heure, X, DF_Apprentissage, "Meilleure_Heure", "classification")


Meilleure_Heure Random Forest Classifier
Modèle : Random Forest Classifier, Type Modèle : classification
 precision/ Accuracy : 0.5645
Temps d'exécution : 1.90 secondes
----------------------------------------

Meilleure_Heure Gradient Boosting (Hist Classif)
Modèle : Gradient Boosting (Hist Classif), Type Modèle : classification
 precision/ Accuracy : 0.5646
Temps d'exécution : 1.05 secondes
----------------------------------------
model save


{'Random Forest Classifier': 0.564499694314245,
 'Gradient Boosting (Hist Classif)': 0.5645676244820325}

### Emplacement dans le panier

#### Explication

Pour cette parti nous allons predire avec notre model l'emplacement moyen de chaque items dans le panier, cela nous permettra de savoir l'emplacement ideal dans le magasin avec les autes parametres.

#### Code

In [42]:
# On defini notre DF qui contient nos informations d'apprentisage
DF_Apprentissage = CSV_untouch.groupby(["product_name"]).agg(
    order_number_max=('order_number', 'max'),
    order_number_min=('order_number', 'min'),
    order_number_mean=('order_number', 'mean'),
    
    dow_mean=('order_dow', 'mean'),
    order_hour_of_day=('order_hour_of_day', "mean"),    
    nombre_sell=('order_id', 'count'),
    reorder_rate=('reordered', 'mean'),
    department=('department', 'first'),
    aisle=('aisle', 'first')
).reset_index() 



# on defini notre DF de reponse qui contient les heures
DF_Reponse = CSV_untouch.groupby(["product_name"]).agg(
    add_to_cart_order=('add_to_cart_order', 'mean'),
).reset_index()

In [43]:
# on defini nos X et Y qui vont etre utilise pour separer nos donnes
X = DF_Apprentissage[['order_number_max', 'order_number_min', 'order_number_mean', 'dow_mean', 'order_hour_of_day', 'nombre_sell', 'reorder_rate', 'department', 'aisle']]
y = DF_Reponse['add_to_cart_order']

# on defini ici toute nos valeurs d'entraienement et de test
X_train_panier, X_test_panier, y_train_panier, y_test_panier = train_test_split(X, y, test_size=0.3, random_state=42)

In [44]:
# on va envoye nos donnes dans la pipeline pour avoir le meilleur resultat
Test_Model(X_train_panier, X_test_panier, y_train_panier, y_test_panier, X, DF_Apprentissage, "Emplacement_panier", "regression")


Emplacement_panier Régression Logistique (Reg)
Modèle : Régression Logistique (Reg), Type Modèle : regression
 MAE (Erreur position) : 2.06
 Score R2              : 0.0989
Temps d'exécution : 0.53 secondes
----------------------------------------

Emplacement_panier Random Forest Regressor
Modèle : Random Forest Regressor, Type Modèle : regression
 MAE (Erreur position) : 2.09
 Score R2              : 0.0778
Temps d'exécution : 2.45 secondes
----------------------------------------

Emplacement_panier Gradient Boosting (Hist)
Modèle : Gradient Boosting (Hist), Type Modèle : regression
 MAE (Erreur position) : 2.06
 Score R2              : 0.0881
Temps d'exécution : 0.49 secondes
----------------------------------------
model save


{'Régression Logistique (Reg)': 0.09891353131463398,
 'Random Forest Regressor': 0.07781182263029163,
 'Gradient Boosting (Hist)': 0.08811728521022333}

### Meilleure jour

#### Explication

Ici nous allons essayer de predire le meilleur jour de vente possible, donc permettre de savoir le jour ou il y a le plus de vente de tel ou tel items.

#### Code

In [20]:
# on defini nos column d'apprentissage
DF_Apprentissage = CSV_untouch.groupby(["product_name"]).agg(
    order_hour_of_day=('order_hour_of_day', "mean"),
    add_to_cart_order=('add_to_cart_order', "mean"),
    nombre_sell=('order_id', 'count'),
    number_mean=('order_number', "mean"),
    reorder_rate=('reordered', 'mean'),
    department=('department', 'first'),
    aisle=('aisle', 'first')
).reset_index() 


# on defini notre DF de reponse qui contient nos jours
DF_Reponse = CSV_untouch.groupby("product_name")['order_dow'].agg(
    lambda x: x.mode()[0]
).reset_index()

In [21]:
# on defini nos X et Y qui vont etre utilise pour separer nos donnes
X = DF_Apprentissage[['order_hour_of_day', 'add_to_cart_order', 'nombre_sell', 'number_mean', 'reorder_rate', 'department', 'aisle']]
y = DF_Reponse['order_dow']


# on defini ici toute nos valeurs d'entraienement et de test
X_train_jour, X_test_jour, y_train_jour, y_test_jour = train_test_split(X, y, test_size=0.3, random_state=42)

In [28]:
# on envoie nos donness dans la pipeline
Test_Model(X_train_jour, X_test_jour, y_train_jour, y_test_jour, X, DF_Apprentissage, "Meilleur Jour", "classification")


Meilleur Jour Régression Logistique (Classif)
Modèle : Régression Logistique (Classif), Type Modèle : classification
 precision/ Accuracy : 0.3950
Temps d'exécution : 9.49 secondes
----------------------------------------

Meilleur Jour Random Forest Classifier
Modèle : Random Forest Classifier, Type Modèle : classification
 precision/ Accuracy : 0.4043
Temps d'exécution : 1.54 secondes
----------------------------------------

Meilleur Jour KNN Classifier
Modèle : KNN Classifier, Type Modèle : classification
 precision/ Accuracy : 0.3560
Temps d'exécution : 0.74 secondes
----------------------------------------

Meilleur Jour Gradient Boosting (Hist Classif)
Modèle : Gradient Boosting (Hist Classif), Type Modèle : classification
 precision/ Accuracy : 0.4017
Temps d'exécution : 2.01 secondes
----------------------------------------
model save


{'Régression Logistique (Classif)': 0.3950139256843964,
 'Random Forest Classifier': 0.4043203586712859,
 'KNN Classifier': 0.3560220093743632,
 'Gradient Boosting (Hist Classif)': 0.40173901229536035}

### Quantite ecoulé

#### Explications:

Pour cette par3tie nous allons regarder le nombre de produits ecoule donc vendu par produit et fournir la quantite  vendu.
Cela permet de savoir a court temps quel sont els produits les plus demandes ou tendance.

#### Code:

In [33]:
# on va definir notre DF deux en 1, donc avec les column d'entrainement et d'apprentissage
df_quantite = CSV_untouch.groupby(['product_name']).agg(
    quantite_ecouler=('order_id', 'count'),
    department=('department', 'first'),    
    aisle=('aisle', 'first'),                       
    reorder_rate=('reordered', 'mean'),              
    position_panier=('add_to_cart_order', 'mean')   
).reset_index()

In [34]:
# on defini nos data dans X et Y pour les separes
X = df_quantite[['department', 'aisle', 'reorder_rate', 'position_panier']]
y = df_quantite['quantite_ecouler']

# on separe nos valeurs pour l'entrainement
X_train_quantite, X_test_quantite, y_train_quantite, y_test_quantite = train_test_split(X, y, test_size=0.3, random_state=42)

In [35]:
# on envoie nos data dans la pipeline
Test_Model(X_train_quantite, X_test_quantite, y_train_quantite, y_test_quantite, X,df_quantite,"Quantite_ecouler", "regression")


Quantite_ecouler Régression Logistique (Reg)
Modèle : Régression Logistique (Reg), Type Modèle : regression
 MAE (Erreur position) : 417.95
 Score R2              : 0.0750
Temps d'exécution : 0.35 secondes
----------------------------------------

Quantite_ecouler Random Forest Regressor
Modèle : Random Forest Regressor, Type Modèle : regression
 MAE (Erreur position) : 287.73
 Score R2              : 0.3860
Temps d'exécution : 1.32 secondes
----------------------------------------

Quantite_ecouler Gradient Boosting (Hist)
Modèle : Gradient Boosting (Hist), Type Modèle : regression
 MAE (Erreur position) : 306.08
 Score R2              : 0.3577
Temps d'exécution : 0.22 secondes
----------------------------------------
model save


{'Régression Logistique (Reg)': 0.07500951415405555,
 'Random Forest Regressor': 0.38595039700282896,
 'Gradient Boosting (Hist)': 0.3577152382251143}

### Meilleur item avec

#### Explication

Nous allons prendre pour chaque item, l'item qui se vend le mieux avec lui.

##### /!\ Attention, pour celui ci nous avon essaye d'iverse technique, metjode, famille et composentes pour obtenir un resultat.
Nous n'avons pas reussi a obtenir un resultat satifsfaisant qui pourrait etre qualifie de non aleatoire, sklearn dans notre configuration et avec le nombre de donnees ainsi que nos machines ne nous laisse tout simplement pas la capacite de realise cette tache en IA, nous utiliserons donc le CSV pour le predire. /!\

( A noter que ici nous ne montrerons que les test non concluant le code pour obtenir les Duos d'item est disponiblr dans le streamlit ou alors a la fin du fichier ``notebook.ipynb``)

#### Code

In [6]:
# on copy notre DF
DF_TP = CSV_untouch.copy()
# on le tri pour etre sur
DF_TP = DF_TP.sort_values(['order_id', 'add_to_cart_order'])

# on utilise .shift(-1) qui permnet de prendre l'element d'apres
DF_TP['produit_suivant'] = DF_TP['product_name'].shift(-1)

# on va creer un df_seq qui permet de prendre le prochain item si et seulement si on est toujours dans le meme user_id sinon on va se retrouver avec un decallage
df_seq = DF_TP[DF_TP['order_id'] == DF_TP['order_id'].shift(-1)]
# on retirer les nan
df_seq = df_seq.dropna(subset=['produit_suivant', 'product_name'])

# on ne prend que 10000 data pour pas tout faire crash
df_seq = df_seq.sample(n=10000, random_state=42)

# on defini nos data
X = df_seq[['product_name']]
y = df_seq['produit_suivant']



In [7]:
X_train_best_item, X_test_best_item, y_train_best_item, y_test_best_item = train_test_split(X, y, test_size=0.3, random_state=42)

In [1]:
Test_Model(X_train_best_item, X_test_best_item, y_train_best_item, y_test_best_item, X, df_seq, "Meilleur_Item_Suivant", "classification" )

NameError: name 'Test_Model' is not defined

# Pipeline

## Explication

Pour obtenir un resultats nous allons utiliser une pipeline, une pipeline va nous servir de guide pour tester automatiquement plusieurs elements et trouver le plus adapter selon la situation.

Nous allons par exemple tester different models ci dessous (Lst_models) afin de voir le plus performant pour nos 6 predictions.


Notre Pipeline test va tester automatiquement nos donnees sur plusieurs model et telecharger automatiquement celui avec le meilleur resultat.

## Code

### Listes de nos models

In [3]:
# on va ici mettre la liste de nos model a tester pour chaque prediction

# on va mettre nos model de regression
Lst_models_Rression = {
    "Régression Logistique (Reg)": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1),
    "Gradient Boosting (Hist)": HistGradientBoostingRegressor(max_iter=200, random_state=42)
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5)
}

# on va mettre nos model de classification
Lst_models_Rression_classification = {
    "Régression Logistique (Classif)": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    "KNN Classifier": KNeighborsClassifier(n_neighbors=5),
    "Gradient Boosting (Hist Classif)": HistGradientBoostingClassifier(max_iter=100, random_state=42)
}

In [9]:
# Toute cette cellule va servir a transformer nos donnes textuel en donnes chiffre et a mettre toute nos donnes sous le bon format pour la pipeline



# on va creer un proceseur qui sert juste a preparer nos donnes
preprocessor = ColumnTransformer(
    transformers=[
        # on prend ici nos valeurs numerique, on va les rendre conforme avec la creatio de column parfaite
        ('num', MaxAbsScaler(), make_column_selector(dtype_include='number')),
        
        # on fait de meme ici pour les valeurs textuels ou autre que numerique, `handle_unknown` evite certaines erreur de column inconnu, `sparse_output` tres rapide mais non reconnu par certains model
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), make_column_selector(dtype_include='object'))
    ]
)


In [5]:

# Notre fonction de pipeline principale
# on creer une fonction
def Test_Model(X_train, X_test, y_train, y_test, X_complet, df_source, Prection_name, type_model):
    # dictionnaire qui va stocker le score de chaque model
    result = {}
    
    # on va definir notre liste de model en fonction de notre type de model souhaiter, si c'est regression ou classification
    Lst_models = Lst_models_Rression if type_model == "regression" else Lst_models_Rression_classification

    # on defini ici nos variables indispensable pour choisir le meilleur model
    best_score = -50 
    best_pipeline = None
    best_name = ""

    
    # on va parcourir notre Liste de model, on va prendre le nom ainsi que le model ci dessus
    for name, modele in Lst_models.items():

        print("")
        print(Prection_name, name)
        
        # on creer la pipeline et on met a l'echelle les donnees
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor), # 1) on normalise les donnees avec notre "processeur" creeer juste avant
            ('classifier', modele)        # 2) on setup la model/ la prediction
        ])

        # on va calculer le temps d'execution
        start_time = time.time()
        
        # on entraine la pipeline avec nos donnees
        pipeline.fit(X_train, y_train)
       
        # on evalue le score et le temps
        predictions = pipeline.predict(X_test)
        temps_execution = time.time() - start_time

        print(f"Modèle : {name}, Type Modèle : {type_model}")
        
        # Si notre type de model est un model de regression:
        if type_model == "regression":
            mae = mean_absolute_error(y_test, predictions)
            score = r2_score(y_test, predictions)
            result[name] = score
            print(f" MAE (Erreur position) : {mae:.2f}")
            print(f" Score R2              : {score:.4f}")

        
        # Si notre model est un model de classification:
        elif type_model == "classification":
            score = accuracy_score(y_test, predictions)
            result[name] = score
            print(f" precision/ Accuracy : {score:.4f}")

        
        # ici on verifie qui a le meilleur score entre le model actuel et le meilleur
        if score > best_score:
            best_score = score
            best_pipeline = pipeline
            best_name = name


        print(f"Temps d'exécution : {temps_execution:.2f} secondes")
        print(f"-"*40)

    # si on a un model alors:
    if best_pipeline is not None:
        # on defini toute nos predictions avec nos column et notre model
        tt_les_predictions = best_pipeline.predict(X_complet)
        # on fait un dictionanire de reponse qui contient en cle le product_name  et en valeur la valeur du model associe au produit
        dico_reponse = dict(zip(df_source['product_name'], tt_les_predictions))

        # on defini ici l'entttdroit et le nom du model (fichier)
        Nom_FIle = f"Model_IA/{Prection_name}_Model"
        # on telecharge en format joblib
        joblib.dump(dico_reponse, Nom_FIle)
        print("model save")
    return result